# Intuitive Physics — Real vs. Reversed Video Classification

Can predicted brain activation patterns (via TRIBE v2) distinguish real physical events from time-reversed versions of the same clips?

**Requirements:** Free Google Colab with T4 GPU (Runtime > Change runtime type > T4 GPU)

**Time estimate:** ~6-10 hours for 100 videos on T4. The experiment caches progress to Google Drive, so if the session disconnects you can just re-run and it resumes.

---

### How to use this notebook

1. Click **Runtime → Change runtime type → T4 GPU** (if not already set)
2. Run each cell in order (Shift+Enter)
3. You'll be prompted to authenticate with Google Drive and HuggingFace
4. Once the experiment starts, you can leave the tab open and come back later

## 0. Verify GPU is available

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

## 1. Mount Google Drive (persistent storage)

Everything is saved to Google Drive so you don't lose progress if the session disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/insilico-physics"
!mkdir -p "{DRIVE_ROOT}"
print(f"Persistent storage: {DRIVE_ROOT}")

## 2. Install dependencies

In [ ]:
!pip install --upgrade pip

# Clone tribev2
!git clone https://github.com/facebookresearch/tribev2.git /content/tribev2-repo 2>/dev/null || \
    echo "tribev2 repo already cloned"

# Install tribev2 (requires torch>=2.5.1 and numpy==2.2.6)
!pip install -e "/content/tribev2-repo[plotting]"

# Clone your project
!git clone https://github.com/cuevase/insilico.git /content/insilico 2>/dev/null || \
    (cd /content/insilico && git pull)
!pip install -r /content/insilico/requirements.txt

# System deps
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!pip install -q gsutil

print("\n\n=== ALL DEPENDENCIES INSTALLED ===")

In [ ]:
# Colab needs a runtime restart after upgrading torch/numpy.
# This cell will crash the runtime on purpose -- that's expected!
# After it restarts, just continue running from the NEXT cell (don't re-run install).
import os
os.kill(os.getpid(), 9)

In [ ]:
from tribev2 import TribeModel
print("tribev2 OK")

import torch
print(f"torch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

import subprocess
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(f"ffmpeg OK: {result.stdout.splitlines()[0]}")

print("\nAll checks passed!")

## 3. Authenticate with HuggingFace

TRIBE v2 uses the gated LLaMA 3.2 model. You need a HuggingFace token:

1. Go to https://huggingface.co/settings/tokens
2. Create a token ("Read" access is enough)
3. Paste it below when prompted

In [ ]:
from huggingface_hub import login
login()

## 4. Download Physics-IQ videos

Downloads test videos from the Physics-IQ benchmark (Google DeepMind). ~Several GB.

Saved to Google Drive so you only download once.

In [ ]:
import os
from pathlib import Path

VIDEO_DIR = Path(DRIVE_ROOT) / "physics-iq-data" / "split-videos" / "testing" / "30FPS"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

existing = list(VIDEO_DIR.glob("*.mp4"))
if len(existing) > 10:
    print(f"Videos already downloaded ({len(existing)} files). Skipping.")
else:
    print("Downloading Physics-IQ test videos (this takes ~10-20 min)...")
    !gsutil -m rsync -r \
        "gs://physics-iq-benchmark/split-videos/testing/30FPS/" \
        "{VIDEO_DIR}/"

    !gsutil cp "gs://physics-iq-benchmark/descriptions.csv" \
        "{Path(DRIVE_ROOT) / 'physics-iq-data' / 'descriptions.csv'}" 2>/dev/null || true

    downloaded = list(VIDEO_DIR.glob("*.mp4"))
    print(f"\nDownloaded {len(downloaded)} videos.")

## 5. Prepare stimuli (real + reversed videos)

Selects 50 center-perspective take-1 videos, copies them, and creates time-reversed versions with ffmpeg.

In [ ]:
import csv
import random
import shutil
import subprocess
from pathlib import Path

STIMULI_DIR = Path(DRIVE_ROOT) / "stimuli"
REAL_DIR = STIMULI_DIR / "real"
REVERSED_DIR = STIMULI_DIR / "reversed"
CSV_PATH = STIMULI_DIR / "stimuli.csv"

N_VIDEOS = 50

if CSV_PATH.exists():
    import pandas as pd
    df = pd.read_csv(CSV_PATH)
    print(f"Stimuli already prepared ({len(df)} rows). Skipping.")
    print(df.head())
else:
    REAL_DIR.mkdir(parents=True, exist_ok=True)
    REVERSED_DIR.mkdir(parents=True, exist_ok=True)

    all_mp4 = sorted(VIDEO_DIR.glob("*.mp4"))
    print(f"Found {len(all_mp4)} total videos")

    # Filter to center-perspective, take-1
    filtered = []
    for p in all_mp4:
        name = p.stem.lower()
        is_center = "perspective-center" in name or "center" in name
        is_take1 = "take-1" in name
        is_testing = "testing" in name
        if is_center and is_take1:
            filtered.append(p)
        elif is_center and is_testing:
            filtered.append(p)

    if not filtered:
        print("No center-perspective take-1 videos found, using all videos.")
        filtered = all_mp4

    # Deduplicate by scenario ID
    unique = {}
    for p in filtered:
        scenario_id = p.stem.split("_")[0]
        if scenario_id not in unique:
            unique[scenario_id] = p
    filtered = sorted(unique.values(), key=lambda p: p.stem)
    print(f"Filtered to {len(filtered)} unique scenarios")

    random.seed(42)
    n = min(N_VIDEOS, len(filtered))
    selected = sorted(random.sample(filtered, n), key=lambda p: p.stem)
    print(f"Selected {len(selected)} videos")

    # Descriptions file for categories
    desc_path = Path(DRIVE_ROOT) / "physics-iq-data" / "descriptions.csv"

    rows = []
    stim_id = 1

    print("\nCopying real videos (stripping audio)...")
    for i, src in enumerate(selected):
        dst = REAL_DIR / src.name
        if not dst.exists():
            subprocess.run(
                ["ffmpeg", "-y", "-i", str(src), "-an", "-c:v", "copy", str(dst)],
                capture_output=True, check=False, timeout=60,
            )
            if not dst.exists():
                shutil.copy2(src, dst)
        print(f"  [{i+1}/{len(selected)}] {src.name}", end="\r")

        name_lower = src.stem.lower()
        cat = "unknown"
        for c in ["fluid", "solid", "optics", "thermo", "magnet"]:
            if c in name_lower:
                cat = c
                break
        rows.append({"id": stim_id, "label": "real", "video_path": f"real/{src.name}", "category": cat})
        stim_id += 1

    print(f"\n\nReversing videos (this takes ~5-15 min)...")
    for i, src in enumerate(selected):
        rev_name = src.stem + "_reversed.mp4"
        dst = REVERSED_DIR / rev_name
        if not dst.exists():
            result = subprocess.run(
                ["ffmpeg", "-y", "-i", str(src),
                 "-vf", "reverse", "-an",
                 "-c:v", "libx264", "-preset", "fast", "-crf", "18",
                 str(dst)],
                capture_output=True, check=False, timeout=120,
            )
        if dst.exists():
            name_lower = src.stem.lower()
            cat = "unknown"
            for c in ["fluid", "solid", "optics", "thermo", "magnet"]:
                if c in name_lower:
                    cat = c
                    break
            rows.append({"id": stim_id, "label": "reversed", "video_path": f"reversed/{rev_name}", "category": cat})
            stim_id += 1
        print(f"  [{i+1}/{len(selected)}] {rev_name}", end="\r")

    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "label", "video_path", "category"])
        writer.writeheader()
        writer.writerows(rows)

    n_real = sum(1 for r in rows if r["label"] == "real")
    n_rev = sum(1 for r in rows if r["label"] == "reversed")
    print(f"\n\nDone: {len(rows)} stimuli ({n_real} real + {n_rev} reversed)")
    print(f"CSV saved to {CSV_PATH}")

## 6. Load TRIBE v2 model

First run downloads model weights (~several GB). Cached to Google Drive for future sessions.

In [ ]:
import sys
sys.path.insert(0, "/content/insilico")

from tribev2 import TribeModel

MODEL_CACHE = str(Path(DRIVE_ROOT) / "model-cache")
Path(MODEL_CACHE).mkdir(parents=True, exist_ok=True)

print("Loading TRIBE v2 (first run downloads weights, ~5-10 min)...")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=MODEL_CACHE)
print("Model loaded!")

## 7. Generate brain predictions

Passes each video through TRIBE v2 → predicted fMRI activation.

**This is the slow step** (~3-6 min per video on T4). Progress is cached per-video to Google Drive, so if the session disconnects just re-run this cell.

For 100 videos: ~6-10 hours total.

In [ ]:
import time
import numpy as np
import pandas as pd

RESULTS_DIR = Path(DRIVE_ROOT) / "results"
CACHE_DIR = RESULTS_DIR / "embeddings"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

stimuli = pd.read_csv(CSV_PATH)
print(f"Loaded {len(stimuli)} stimuli")
print(f"  Real: {(stimuli.label == 'real').sum()}, Reversed: {(stimuli.label == 'reversed').sum()}")

vectors = []
labels = []
valid_ids = []
skipped = []

total = len(stimuli)
t_start = time.time()

for i, row in stimuli.iterrows():
    idx = int(i) + 1
    stim_id = row.id
    cache_path = CACHE_DIR / f"stim_{stim_id:04d}.npy"

    # Load from cache if already computed
    if cache_path.exists():
        vec = np.load(cache_path)
        vectors.append(vec)
        labels.append(1 if row.label == "reversed" else 0)
        valid_ids.append(stim_id)
        print(f"  [{idx}/{total}] CACHED: {row.video_path}")
        continue

    video_path = str(STIMULI_DIR / row.video_path)
    if not Path(video_path).exists():
        print(f"  [{idx}/{total}] MISSING: {video_path}")
        skipped.append(stim_id)
        continue

    t0 = time.time()
    try:
        df = model.get_events_dataframe(video_path=video_path)
        preds, _ = model.predict(events=df)
        vec = preds.mean(axis=0) if preds.ndim == 2 else preds
        np.save(cache_path, vec)

        vectors.append(vec)
        labels.append(1 if row.label == "reversed" else 0)
        valid_ids.append(stim_id)

        elapsed = time.time() - t0
        total_elapsed = (time.time() - t_start) / 60
        done = len(vectors)
        remaining = total - idx
        avg_per = total_elapsed / done if done > 0 else 0
        eta = avg_per * remaining

        print(f"  [{idx}/{total}] {row.label}: {row.video_path} "
              f"({elapsed:.0f}s, {vec.shape[0]} vertices) "
              f"[{total_elapsed:.0f}min elapsed, ~{eta:.0f}min remaining]")

    except Exception as e:
        print(f"  [{idx}/{total}] ERROR: {e}")
        skipped.append(stim_id)

X = np.vstack(vectors)
y = np.array(labels)

total_time = (time.time() - t_start) / 60
print(f"\nDone: {X.shape[0]} embeddings in {total_time:.1f} minutes")
print(f"Shape: {X.shape}")
if skipped:
    print(f"Skipped: {len(skipped)} stimuli")

np.save(RESULTS_DIR / "embeddings_X.npy", X)
np.save(RESULTS_DIR / "labels_y.npy", y)
print("Saved embeddings to Google Drive.")

## 7b. Free GPU memory

The TRIBE v2 model uses most of the RAM. We need to unload it before running the classifier.

In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory freed. RAM available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

## 8. Train classifier & evaluate

L1-regularized logistic regression with 5-fold stratified cross-validation.

In [ ]:
import json
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

# If resuming a session, reload from Drive
if "X" not in dir() or X is None:
    X = np.load(RESULTS_DIR / "embeddings_X.npy")
    y = np.load(RESULTS_DIR / "labels_y.npy")

N_FOLDS = 5
REG_C = 1.0

print(f"Classification: {X.shape[0]} samples, {X.shape[1]} features, {N_FOLDS}-fold CV")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = LogisticRegression(C=REG_C, max_iter=5000, solver="saga", penalty="l1")
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

y_pred = cross_val_predict(clf, X_scaled, y, cv=cv)
y_prob = cross_val_predict(clf, X_scaled, y, cv=cv, method="predict_proba")[:, 1]

acc = accuracy_score(y, y_pred)
auc = roc_auc_score(y, y_prob)
cm = confusion_matrix(y, y_pred)

print(f"\n{'='*50}")
print(f"  Accuracy:  {acc:.1%}")
print(f"  ROC AUC:   {auc:.3f}")
print(f"{'='*50}")
print(f"\n{classification_report(y, y_pred, target_names=['real', 'reversed'])}")

clf.fit(X_scaled, y)

results = {
    "accuracy": float(acc),
    "roc_auc": float(auc),
    "confusion_matrix": cm.tolist(),
    "n_samples": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "n_folds": N_FOLDS,
}

with open(RESULTS_DIR / "classification_results.json", "w") as f:
    json.dump(results, f, indent=2)

cortical_weights = clf.coef_[0][:20484]  # 2 hemispheres x 10242 vertices
np.save(RESULTS_DIR / "classifier_weights.npy", cortical_weights)

print(f"\nResults saved to Google Drive.")

## 9. Generate figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Brain surface maps ---
try:
    from nilearn import datasets, plotting

    fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")
    n_hemi = 10_242
    vmax = float(np.percentile(np.abs(cortical_weights), 95))

    for hemi, hemi_name in [("left", "Left"), ("right", "Right")]:
        for view in ["lateral", "medial"]:
            fig, ax = plt.subplots(1, 1, figsize=(6, 5), subplot_kw={"projection": "3d"})
            data = cortical_weights[:n_hemi] if hemi == "left" else cortical_weights[n_hemi:]
            plotting.plot_surf_stat_map(
                fsaverage[f"pial_{hemi}"], data,
                hemi=hemi, view=view, cmap="cold_hot",
                threshold=vmax * 0.2, vmax=vmax,
                axes=ax, colorbar=True,
                bg_map=fsaverage[f"sulc_{hemi}"],
                title=f"Real vs Reversed Physics — {hemi_name} {view.capitalize()}",
            )
            fname = f"weights_{hemi}_{view}.png"
            fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
            plt.close(fig)
            print(f"Saved {fname}")
except Exception as e:
    print(f"Brain figures skipped: {e}")

# --- PCA scatter ---
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
for label, name, color in [(0, "Real", "#2ECC71"), (1, "Reversed", "#E67E22")]:
    mask = y == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=name, alpha=0.7, s=60)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("Brain Response Patterns: Real vs Reversed Physics")
ax.legend()
fig.savefig(FIG_DIR / "pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved pca_scatter.png")

# --- Confusion matrix ---
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Real", "Reversed"])
ax.set_yticklabels(["Real", "Reversed"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix (Acc: {results['accuracy']:.1%})")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha="center", va="center",
                color="white" if cm[i][j] > cm.max() / 2 else "black", fontsize=16)
fig.colorbar(im)
fig.savefig(FIG_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

## 10. Summary & download results

In [ ]:
print("=" * 60)
print("  EXPERIMENT COMPLETE")
print(f"  Accuracy: {results['accuracy']:.1%}  |  AUC: {results['roc_auc']:.3f}")
print("=" * 60)
print(f"\nAll results saved to Google Drive at:")
print(f"  {RESULTS_DIR}")
print(f"\nFiles:")
for f in sorted(RESULTS_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(RESULTS_DIR)}  ({size_mb:.1f} MB)")

In [ ]:
# Zip results for easy download
import shutil

zip_path = Path(DRIVE_ROOT) / "physics_results"
shutil.make_archive(str(zip_path), "zip", str(RESULTS_DIR))
print(f"Results zipped to: {zip_path}.zip")
print(f"\nDownload from Google Drive: MyDrive/insilico-physics/physics_results.zip")
print("Or download directly from Colab using the file browser on the left.")